In [1]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import simulator, graph_utils
import ipywidgets as widgets
from ipywidgets import interact


In [2]:
# 1. Define interactive widgets for graph setup
nodes_widget = widgets.IntSlider(min=10, max=100, step=5, value=30, description='Nodes (N):')
p_widget = widgets.FloatSlider(min=0.05, max=0.5, step=0.05, value=0.1, description='Edge Prob (p):')
beta_widget = widgets.FloatSlider(min=0.05, max=1.0, step=0.05, value=0.3, description='Beta (Infect):')
gamma_widget = widgets.FloatSlider(min=0.01, max=0.5, step=0.01, value=0.1, description='Gamma (Recov):')

# Display widgets in the cell
display(nodes_widget, p_widget, beta_widget, gamma_widget)

IntSlider(value=30, description='Nodes (N):', min=10, step=5)

FloatSlider(value=0.1, description='Edge Prob (p):', max=0.5, min=0.05, step=0.05)

FloatSlider(value=0.3, description='Beta (Infect):', max=1.0, min=0.05, step=0.05)

FloatSlider(value=0.1, description='Gamma (Recov):', max=0.5, min=0.01, step=0.01)

In [3]:
# 2. Extract values from widgets
n_nodes = nodes_widget.value
p_edge = p_widget.value
beta = beta_widget.value
gamma = gamma_widget.value

# 3. Generate graph and lock positions
graph = graph_utils.generate_random_graph(n=n_nodes, p=p_edge)
pos = nx.spring_layout(graph, seed=42)  # Layout locked!

# 4. Initialize and run simulation
sim = simulator.RumorSimulator(graph=graph, beta=beta, gamma=gamma)
sim.initialize(infected_nodes=[0])
history = sim.run(steps=30)

print(f"Graph generated with {len(graph.nodes())} nodes. Simulation finished for {len(history)} steps!")

Graph generated with 30 nodes. Simulation finished for 30 steps!


In [4]:
from ipywidgets import interact

@interact(t=widgets.IntSlider(min=0, max=len(history) - 1, step=1, value=0, description='Time Tick (t):'))
def render_step(t):
    graph_utils.visualize_graph(graph, status=history[t], pos=pos)

interactive(children=(IntSlider(value=0, description='Time Tick (t):', max=29), Output()), _dom_classes=('widg…

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import simulator, graph_utils
import ipywidgets as widgets
from ipywidgets import interact, interactive_output, VBox, HBox

# ==========================================
# 1. Define Interactive Widgets
# ==========================================
model_widget = widgets.Dropdown(
    options=[('Erdős–Rényi (ER)', 'ER'), ('Barabási–Albert (BA)', 'BA')],
    value='BA',
    description='Graph Model:'
)

nodes_widget = widgets.IntSlider(min=10, max=100, step=5, value=30, description='Nodes (N):')
p_widget = widgets.FloatSlider(min=0.05, max=0.5, step=0.05, value=0.1, description='Edge Prob (p):')
m_widget = widgets.IntSlider(min=1, max=10, step=1, value=2, description='Edges (m):')

beta_widget = widgets.FloatSlider(min=0.05, max=1.0, step=0.05, value=0.3, description='Beta (Infect):')
gamma_widget = widgets.FloatSlider(min=0.01, max=0.5, step=0.01, value=0.1, description='Gamma (Recov):')

# Dynamically enable/disable parameters based on model selected
def on_model_change(change):
    if change['new'] == 'ER':
        p_widget.disabled = False
        m_widget.disabled = True
    else:
        p_widget.disabled = True
        m_widget.disabled = False

model_widget.observe(on_model_change, names='value')
on_model_change({'new': model_widget.value})  # Set initial state

display(VBox([
    model_widget,
    HBox([nodes_widget, p_widget, m_widget]),
    HBox([beta_widget, gamma_widget])
]))

In [ ]:
# ==========================================
# 2. Extract Values & Enforce Constraints
# ==========================================
model_type = model_widget.value
n_nodes = nodes_widget.value
p_edge = p_widget.value
m_edges = m_widget.value
beta = beta_widget.value
gamma = gamma_widget.value

# Constraint validation for Barabási–Albert: 1 <= m < n
if model_type == 'BA':
    if m_edges >= n_nodes:
        m_edges = n_nodes - 1
        print(f"⚠️ Adjusted m to {m_edges} to satisfy 1 <= m < n constraint.")

# ==========================================
# 3. Generate Graph & Analyze Hubs
# ==========================================
if model_type == 'ER':
    graph = graph_utils.generate_random_graph(n=n_nodes, p=p_edge)
else:
    graph = graph_utils.generate_barabasi_albert_graph(n=n_nodes, m=m_edges)

# Calculate layout (fixed positions across steps)
pos = nx.spring_layout(graph, seed=42)

# Spot the Hubs / Compare Degrees
degrees = dict(graph.degree())
max_node = max(degrees, key=degrees.get)
avg_degree = np.mean(list(degrees.values()))

print(f"📊 Model: {model_type}")
print(f"• Total Nodes: {len(graph.nodes())}")
print(f"• Total Edges: {len(graph.edges())}")
print(f"• Average Degree: {avg_degree:.2f}")
print(f"🔥 Highest Degree Node: Node {max_node} with {degrees[max_node]} connections")

In [ ]:
# ==========================================
# 4. Run Simulation & Interactive Visualizer
# ==========================================
sim = simulator.RumorSimulator(graph=graph, beta=beta, gamma=gamma)
sim.initialize(infected_nodes=[max_node])  # Seed infection at the largest hub!
history = sim.run(steps=30)

print(f"Simulation completed for {len(history)} time steps starting at Hub Node {max_node}.\n")

@interact(t=widgets.IntSlider(min=0, max=len(history) - 1, step=1, value=0, description='Time Tick (t):'))
def render_step(t):
    graph_utils.visualize_graph(graph, status=history[t], pos=pos)